In [ ]:
#@title 按這裡開始（先按 ▶）
print("✅ W13 出發！本週目標：用 Whisper 轉錄自己的錄音，並自己寫出算錯字率的程式")
print("本週不需要 GPU，base 模型在 CPU 就跑得動")
print("本週要自己補五個空：CER 兩行、差異比對兩處、批次轉錄的檔案樣式")

# W13　內建聽寫與 Whisper 大對決（電腦教室版）

**本週的學習目標是「自己寫出算錯字率的程式」，不是「把最大的模型跑起來」。**
模型下載太久就換 `tiny`，一樣拿得到全部分數。

**動手順序**
1. 用電腦的「錄音機」錄三段（`quiet`、`noisy`、`accent`），同時用 Google 文件的語音輸入打一次。
2. 三個檔案上傳到雲端硬碟的 `AI115` 資料夾。
3. 回到這一本，由上往下執行。

**開始之前**：功能表「檔案 → 在雲端硬碟中儲存副本」，
再**先按下面第 1 格**讓它在背景下載模型，你去做錄音，回來剛好裝好。

**檔名一律用英文**：`quiet.m4a`、`noisy.m4a`、`accent.m4a`，
中文檔名容易出錯，任務五的批次轉錄也讀不到。

### 第 1 格：安裝並載入 Whisper 模型

**這一格要做什麼**：安裝套件、載入 `base` 模型。沒有空格，直接執行。

**寫對了會看到什麼**：印出「載入完成： base ，花了 NN 秒」，
以及「有沒有 GPU： False」——沒有 GPU 是正常的，本週不需要。

**第一次會下載模型檔**，`base` 大約要等一到兩分鐘，
下載中不要按停止、不要重整分頁。等待的時候先去做任務一的錄音，
或先拿紙筆手算一次錯字率。

**全班同時 `pip install` 會塞車**，卡住就等 30 秒重跑一次，不要一直按停止。

In [ ]:
#@title 第 1 格：載入 Whisper 模型（第一次會下載，請耐心等）
!pip install -q openai-whisper
import whisper, torch, time
size = "base"        # tiny 更快、base 較準，先用 base
t0 = time.time()
model = whisper.load_model(size)
print("載入完成：", size, "，花了", round(time.time()-t0), "秒")
print("有沒有 GPU：", torch.cuda.is_available())

### 第 2 格：轉錄你自己的錄音

**這一格要做什麼**：掛載雲端硬碟，只改 `name` 那一行，
三個情境（quiet、noisy、accent）各跑一次。

**寫對了會看到什麼**：印出一段中文，那就是 Whisper 聽到的內容。
每跑完一個情境，先把結果複製到筆記裡，不然換檔名跑下一個就被蓋掉了。

**跳出「連線至 Google 雲端硬碟」要按允許**；
說找不到檔案就回雲端硬碟網頁版確認檔案真的在 `AI115` 裡，再重跑這一格。

`fp16=False` 是因為沒開 GPU，不加會跳一個警告。中文一定要指定 `language="zh"`。

In [ ]:
#@title 第 2 格：轉錄錄音（三個情境各跑一次）
from google.colab import drive
drive.mount("/content/drive")
base = "/content/drive/MyDrive/AI115/"
name = "quiet.m4a"   # 換 noisy.m4a、accent.m4a 再各跑一次
r = model.transcribe(base + name, language="zh", fp16=False)
whisper_text = r["text"].strip()
print(whisper_text)

### 第 3 格：自己寫錯字率 CER

`difflib` 的 `get_opcodes()` 會把兩段文字比成好幾段，
每一段長得像 `('replace', 5, 6, 5, 6)`：第一個是差異的種類
（`equal`、`replace`、`delete`、`insert`），後面四個數字是位置——
`i1:i2` 是原句的範圍，`j1:j2` 是辨識結果的範圍。

**先印出來看**：在函式裡加一行 `print(ops)` 看五分鐘，再回頭寫。

**這一格要做什麼**：補兩行。
- 第一行：把這一段的錯誤字數**累加**起來。三種錯都要算到——打錯字、漏字、多打的字，
  所以要取原句與辨識結果**比較長的那一邊**。
- 第二行：回傳「錯的字數 ÷ 原句字數」，是小數，乘以 100 才是百分比。

**寫對了會看到什麼**：印出一個百分比，而且要跟你手算的那個數字對得起來。
安靜那段最低、吵雜那段最高。

**還沒錄音的人**：這一格最後一行會說 `NameError: whisper_text`，那是正常的，
`cer()` 已經定義好了，直接往下跑「自我檢查」那一格驗證你寫得對不對。

In [ ]:
#@title 第 3 格：自己寫 cer()
import difflib
def cer(ref, hyp):
    ops = difflib.SequenceMatcher(None, ref, hyp).get_opcodes()
    err = 0
    for tag, i1, i2, j1, j2 in ops:
        if tag == "equal":
            continue
        err = ____   # ← 自己寫：累加 max(i2-i1, j2-j1)
    return ____      # ← 自己寫：錯的字數 ÷ 原句字數
ref = "今天天氣很好我們在南臺科技大學電子系上課"
print("錯字率 =", round(cer(ref, whisper_text) * 100, 1), "%")

### 自我檢查：你的 `cer()` 跟上課例題對得起來嗎

上課投影片那張「同一句話，三種情境的錯字率」的例題，答案是 5%、15%、25%。
這一格拿同樣的三句去餵你寫的 `cer()`。

**寫對了會看到什麼**：三行都是 ✅。
出現 ❌ 就回第 3 格改：
- 只算 `i2-i1` 會漏掉「多打的字」，只算 `j2-j1` 會漏掉「漏字」，所以要用 `max()`。
- 分母一定是**原句**的字數 `len(ref)`，不是辨識結果的字數。

In [ ]:
#@title 自我檢查（投影片未含，執行所需）
ref0 = "今天天氣很好我們在南臺科技大學電子系上課"   # 例題原句，20 個字
ex = {"安靜": ("今天天氣很好我們在南台科技大學電子系上課", 5.0),
      "台灣國語": ("今天天氣粉好我們在南台科技大學電子西上課", 15.0),
      "吵雜": ("今天天氣粉好我們在南台科機大學電子細課", 25.0)}
for k, (hyp, ans) in ex.items():
    got = round(cer(ref0, hyp) * 100, 1)
    print(k, "程式算出", got, "%　例題答案", ans, "%",
          "✅" if got == ans else "❌ 回第 3 格再改")

### 第 4 格：自己寫「錯在哪」

**這一格要做什麼**：再補兩個空。
- 第一個：把 `tag`（哪一種差異）、`ref[i1:i2]`（原句這一段）、
  `hyp[j1:j2]`（辨識結果這一段）印出來。
- 第二個：第二次呼叫要換成**內建聽寫**那一段文字。

**寫對了會看到什麼**：印出來的每一行就是一個錯，
行數要跟你算出來的錯字數對得上。兩邊各印一次，
就看得出 Whisper 與內建聽寫錯的字根本不一樣。

**還沒做內建聽寫的人**：先把 `phone_text` 那一行貼上 Google 文件語音輸入的結果。

In [ ]:
#@title 第 4 格：自己寫 show_diff()
phone_text = "把語音輸入打出來的那一段貼在這裡"
def show_diff(ref, hyp):
    m = difflib.SequenceMatcher(None, ref, hyp)
    for tag, i1, i2, j1, j2 in m.get_opcodes():
        if tag == "equal":
            continue
        ____   # ← 自己寫：印出 tag、ref[i1:i2]、hyp[j1:j2]
show_diff(ref, whisper_text)
print("---- 內建聽寫錯在哪 ----")
show_diff(ref, ____)   # ← 自己寫：換成聽寫那一段

### 第 5 格（進階）：批次轉錄成 CSV

三個檔一次跑完，輸出一張表。做不完不影響及格。

**這一格要做什麼**：補上「`AI115` 裡所有 `.m4a` 的路徑樣式」。
提示：`glob` 的萬用字元是 `*`，路徑要接在 `base` 後面。

**寫對了會看到什麼**：一張三列的表（檔案、字數、錯字率、文字），
雲端硬碟裡多一個 `w13_result.csv`，可以下載到電腦貼進報告當證據。

**下載被擋**：點網址列右邊的圖示，選「一律允許」。

In [ ]:
#@title 第 5 格（進階）：批次轉錄
import glob, pandas as pd
pattern = ____   # ← 自己寫：AI115 裡所有 .m4a 的路徑
rows = []
for p in sorted(glob.glob(pattern)):
    out = model.transcribe(p, language="zh", fp16=False)
    t = out["text"].strip()
    rows.append({"檔案": p.split("/")[-1], "字數": len(t),
                 "錯字率": round(cer(ref, t), 3), "文字": t})
df = pd.DataFrame(rows)
df.to_csv(base + "w13_result.csv", index=False)
df

### 收工：延伸挑戰與繳交

- **A**（每組都要做）：把 `size` 從 `base` 改成 `tiny` 再跑一次，
  用你寫的 `cer()` 算出兩個尺寸差幾個百分點。
- **B**：念一段中英夾雜的句子（含 Colab、Whisper），
  看內建聽寫與 Whisper 各錯在哪。
- **C**：說出 Whisper 在吵雜情境為什麼比較穩，用你自己算出來的錯字率當證據。

**常見狀況**：錄音機錄不到聲音＝麥克風權限沒開（設定 → 隱私權 → 麥克風）；
語音輸入是灰的＝不是用 Chrome 開的；
筆記本改了沒存到＝開的是唯讀的 GitHub 版，先「複製到雲端硬碟」再改。

In [ ]:
#@title 收工檢查（直接按 ▶）
print("本週要交：三情境的錄音與辨識結果、錯字率紀錄表（手算與程式各一份）")
print("以及寫完的 .ipynb（要含你自己寫的 cer 函式）")
print("檔名：AI導論_W13_學號_姓名，上傳課程表單，下次上課前一天 23:59")
print("提醒：檔案一律存雲端硬碟，教室電腦重開機會還原")

---

<details>
<summary>參考解（五個空格都自己試過再打開）</summary>

```python
# 第 3 格
        err = err + max(i2 - i1, j2 - j1)
    return err / len(ref)

# 第 4 格
        print(tag, ref[i1:i2], "→", hyp[j1:j2])
show_diff(ref, phone_text)

# 第 5 格
pattern = base + "*.m4a"
```

為什麼是這樣寫：

- `max(i2-i1, j2-j1)` 取原句與辨識結果比較長的那一邊，
  這樣「替換」「漏字」「多字」三種錯都數得到。
  只算其中一邊就會少算一種。
- 分母是**原句**的字數 `len(ref)`；回傳的是小數，所以外面才要乘 100。
- `glob` 的萬用字元 `*` 代表「任何檔名」，`base + "*.m4a"` 就是
  `AI115` 資料夾裡所有的 m4a 檔。

用上課例題驗算（原句 20 字）：
台→臺 1 個錯＝5%；很、臺、系 3 個錯＝15%；4 個錯字加漏 1 字＝25%。
你的程式算出來要跟這三個數字一模一樣。

</details>